# Lab 1 · End-to-End Pipeline Assembly

**Objective:** Integrate retrieval, orchestration, guardrails, and observability layers into a cohesive PoC pipeline ready for Week 8 demos.

## Lab Overview
- **Estimated time:** 150 minutes
- **Personas:** Backend engineer, security engineer, observability/SRE
- **Prerequisites:**
  - Langfuse workspace provisioned during Week 7
  - Guardrail catalog and red-team prompts from `resources/`
  - Vector store seeded with enterprise documents (Week 5)
  - Environment variables defined in `.env` or vault (see checklist)
- **Deliverables:**
  - Working `query -> retrieve -> guardrails -> LLM -> response` pipeline
  - Langfuse traces with custom spans
  - Smoke test evidence (latency, success rate)

## Scenario
You are finalizing the Week 8 PoC for the 
Risk Analyst Assist
 use case. The objective is to surface a concise, policy-safe summary of the top risk escalations for executive review. The pipeline must:
1. Accept persona-aware queries via an API layer.
2. Retrieve relevant documents from the enterprise vector store.
3. Apply guardrails (prompt filtering + output validation).
4. Generate a response with the designated LLM, including fallback logic.
5. Emit structured traces and metrics for observability.

---
### 🧪 Checkpoint Tracker
Mark each task as you complete it.
- [ ] Environment bootstrapped (dependencies, environment variables)
- [ ] Retrieval service returning top-k passages
- [ ] Guardrail policies enforced on inputs and outputs
- [ ] Langfuse spans visible with custom metadata
- [ ] Smoke test suite green (< 3s latency, success rate > 95%)
- [ ] Readiness scorecard updated

## 1. Environment Bootstrap
Validate dependencies, configuration, and secrets. Update `.env` or secret manager entries if values are missing.

In [ ]:
import os
from pathlib import Path

EXPECTED_ENV_KEYS = [
    "OPENAI_API_KEY",
    "LANGFUSE_PUBLIC_KEY",
    "LANGFUSE_SECRET_KEY",
    "LANGFUSE_HOST",
    "VECTOR_DB_URL",
    "VECTOR_DB_TOKEN"
]
missing = [key for key in EXPECTED_ENV_KEYS if not os.getenv(key)]
if missing:
    raise EnvironmentError(f"Missing env vars: {missing}")
else:
    print("Environment configuration looks good.")

> **Tip:** If you are running locally, load variables from `.env` using `python-dotenv` or your preferred secret manager.

## 2. Retrieval Layer Assembly
Wire the ingestion and retrieval logic. Start with the baseline pipeline (Week 5) and extend it with persona-specific filters if applicable.

In [ ]:
from typing import List, Dict
from rag.retriever import HybridRetriever
from rag.chunking import load_corpus

# TODO: parameterize index names (e.g., risk_analyst, compliance_officer)
CORPUS_PATH = Path("data/corpus/risk-escalations.jsonl")
INDEX_NAME = "risk-escalations-v1"

corpus = load_corpus(CORPUS_PATH)
retriever = HybridRetriever(index_name=INDEX_NAME, vector_db_url=os.environ["VECTOR_DB_URL"])

def retrieve_context(query: str, persona: str, k: int = 5) -> List[Dict]:
    """Return enriched passages tailored to the caller persona."""
    filters = {"persona": persona} if persona else None
    results = retriever.search(query=query, top_k=k, filters=filters)
    # Attach metadata for downstream guardrails and analytics
    for item in results:
        item.setdefault("metadata", {})
        item["metadata"]["persona"] = persona
    return results

# Quick smoke check
sample = retrieve_context("Summarize APAC escalations", persona="executive")
print(f"Retrieved {len(sample)} passages")

Document any ingestion or embedding updates in the change log before proceeding.

## 3. Guardrails & Prompt Orchestration
Integrate guardrails that validate both inbound prompts and outbound responses. Use the Week 7 policy YAML as a baseline.

In [ ]:
from guardrails import GuardrailEngine, ValidationError
from guardrails.policies import load_policy

policy = load_policy(Path("config/guardrails/prompt-firewall.yaml"))
guardrail_engine = GuardrailEngine(policy=policy)

def apply_guardrails(prompt: str, context: List[Dict]) -> str:
    """Compose a safe, persona-aware prompt for the LLM."""
    guardrail_engine.validate_input(prompt, metadata={"persona": context[0]["metadata"].get("persona")})
    constructed = guardrail_engine.build_prompt(prompt, context=context)
    return constructed

def validate_output(output: str) -> str:
    try:
        guardrail_engine.validate_output(output)
    except ValidationError as exc:
        # TODO: Decide whether to regenerate, fallback, or escalate to human
        raise
    return output

> **Note:** Capture guardrail decisions (allow/deny, rule IDs) for analytics and stakeholder reporting.

## 4. Orchestration & LLM Invocation
Connect retrieval, guardrails, and the LLM into an orchestrated flow with fallback logic.

In [ ]:
from orchestration.router import LLMRouter, LLMResponse
from orchestration.fallbacks import fallback_strategy
from observability.tracing import pipeline_tracer

router = LLMRouter(primary="gpt-4.1", fallback="gpt-4o-mini")

@pipeline_tracer.trace(name="risk_analyst_pipeline")
def run_pipeline(query: str, persona: str) -> LLMResponse:
    context = retrieve_context(query, persona=persona)
    prompt = apply_guardrails(query, context)
    try:
        response = router.invoke(prompt, context=context)
    except Exception as primary_exc:
        pipeline_tracer.log_event("primary_model_failure", {"error": str(primary_exc)})
        response = fallback_strategy(router, prompt, context=context)
    cleaned = validate_output(response.content)
    return LLMResponse(content=cleaned, metadata=response.metadata)

### Smoke Test the Pipeline

In [ ]:
test_response = run_pipeline("Give me the top three APAC escalations to watch", persona="executive")
print(test_response.content[:400])

Log the smoke test outcome in the risk register if issues surface.

## 5. Observability Instrumentation
Ensure Langfuse (or equivalent) captures the full span graph with key metadata: persona, latency, error status, guardrail decisions.

In [ ]:
from langfuse import Langfuse

langfuse_client = Langfuse()

@pipeline_tracer.on_span_finish
def publish_trace(span):
    langfuse_client.log_trace(
        trace_id=span.id,
        name=span.name,
        metadata={
            "persona": span.tags.get("persona"),
            "latency_ms": span.metrics.get("duration_ms"),
            "guardrail_decision": span.tags.get("guardrail_decision"),
        }
    )
    if span.status == "error":
        langfuse_client.log_event("pipeline_error", {"trace_id": span.id})

> **Verification:** Navigate to Langfuse and confirm the new spans appear under the PoC project with the correct tags. Capture a screenshot for the readiness pack.

## 6. Regression & Readiness Checks
Execute the automated checks referenced in the lessons. Update the readiness scorecard upon completion.

In [ ]:
import subprocess

commands = [
    ["pytest", "tests/smoke", "--maxfail=1", "--disable-warnings", "-q"],
    ["python", "scripts/guardrail_report.py", "--min-block-rate", "0.95"],
    ["python", "scripts/check_latency.py", "--threshold-ms", "3000"]
]
for cmd in commands:
    result = subprocess.run(cmd, check=True, capture_output=True, text=True)
    print(cmd, "✔")
    print(result.stdout)

Record test outputs in `reports/` (commit or attach to ticket).

## 7. Integration Retrospective
Complete the table below with learnings and blockers. Share the summary in the daily brief.

| Item | Observation | Owner | Follow-up |
| ---- | ----------- | ----- | --------- |
| Retrieval quality | TODO |  |  |
| Guardrail bypass attempts | TODO |  |  |
| Latency hot spots | TODO |  |  |
| Cost anomalies | TODO |  |  |
| Telemetry gaps | TODO |  |  |

## Submission Checklist
- [ ] Notebook executed top-to-bottom without errors
- [ ] Readiness scorecard updated
- [ ] Guardrail and load test reports stored in `reports/`
- [ ] Demo storyboard synced with new capabilities
- [ ] Daily brief (Slack) updated with key outcomes

Upload the executed notebook, relevant artifacts, and a short summary to the team repository or project tracker.